In [ ]:
# ==========================================
# 1. Importações
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


In [ ]:
tabela = pd.read_csv('../data/02-processed/risco-de-credito-tratados.csv')

In [ ]:
# ===============================
# 2. Separar X e y
# ===============================

X = tabela.drop('classificação_de_crédito', axis=1)
y = tabela['classificação_de_crédito']

In [ ]:
# ===============================
# 3. Transformar variáveis categóricas
# ===============================

X = pd.get_dummies(X, drop_first=True)

In [ ]:
# ===============================
# 4. Separar treino e teste
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# ===============================
# 5. Criar os modelos
# ===============================

modelos = {

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("modelo", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]),

    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("modelo", XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
            eval_metric="logloss",
            scale_pos_weight=3.58
        ))
    ])
}


In [ ]:
# ===============================
# 6. Treinar e avaliar os modelos
# ===============================

resultados = []

for nome, modelo in modelos.items():

    # ===============================
    # Treinamento
    # ===============================
    modelo.fit(X_train, y_train)

    # ===============================
    # Predições
    # ===============================
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)

    # ===============================
    # Métricas
    # ===============================
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # ===============================
    # Resultados
    # ===============================
    print("=" * 60)
    print(f"{nome}")
    print("=" * 60)

    print("\nMatriz de Confusão:")
    print(confusion_matrix(y_test, y_pred))

    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred))

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    resultados.append({
        "Modelo": nome,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": auc
    })

# ===============================
# DataFrame com resultados
# ===============================
df_resultados = pd.DataFrame(resultados)

print("\n")
print("=" * 60)
print("Resumo Final")
print("=" * 60)

print(df_resultados.sort_values(by="ROC-AUC", ascending=False))

In [ ]:
# ===============================
# 7. Comparar os modelos
# ===============================

df_resultados = pd.DataFrame(resultados)
df_resultados.sort_values(by="ROC-AUC", ascending=False)

In [ ]:
# ===============================
# 8. Feature Importance - Random Forest
# ===============================

rf = modelos["Random Forest"].named_steps["modelo"]

importancias_rf = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importancias_rf.head(15))



In [ ]:
# ===============================
# 9. Feature Importance - XGBoost
# ===============================

xgb = modelos["XGBoost"].named_steps["modelo"]

importancias_xgb = pd.Series(
    xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importancias_xgb.head(15))

In [ ]:
plt.figure(figsize=(10,6))

importancias_rf.head(15).plot(kind='barh')

plt.title('Importância das Variáveis - Random Forest')

plt.gca().invert_yaxis()

plt.show()